In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import pandas as pd
import numpy as np
import pickle 

In [2]:
## Load ANN Train model, sclar pickle and one hotencode pickle 
model = load_model("model.keras")

with open('label_encoder_gender.pkl','rb') as file:
    lable_encoder_gender = pickle.load(file)

with open("onehot_encoder_geography.pkl",'rb') as file:
    onehot_encoder_geography = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\keras\src\saving\saving_lib.py:576: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [3]:
# Example input data
input_data = {
    "CreditScore":600,
    "Geography":"France",
    "Gender":"Male",
    "Age":40,
    "Tenure":3,
    "Balance":60000,
    "NumOfProducts":2,
    "HasCrCard":1,
    "IsActiveMember":1,
    "EstimatedSalary":1
}

input_data = pd.DataFrame([input_data])
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,1


In [8]:
# onehot encode Geography
geo_encoder =onehot_encoder_geography.transform(np.array(input_data["Geography"]).reshape(-1, 1)).toarray()
geo_encoded_df = pd.DataFrame(geo_encoder, columns= onehot_encoder_geography.get_feature_names_out(["Geography"]))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [9]:
# Encode categorical variable
input_data["Gender"] = lable_encoder_gender.transform(input_data["Gender"])
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,1


In [10]:
# Concate onehot encoded data and label encoded data
input_data = pd.concat([input_data.drop("Geography",axis=1), geo_encoded_df], axis=1)
input_data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,1,1.0,0.0,0.0


In [12]:
# Scale input data
input_scale = scaler.transform(input_data)
input_scale


array([[-0.47154541,  0.90911166,  0.09477172, -0.69844549, -0.29010416,
         0.80510537,  0.63367318,  0.95214374, -1.71825611,  0.98019606,
        -0.57581067, -0.56349184]])

In [13]:
## Predict Churn 
prediction = model.predict(input_scale)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 247ms/step


array([[0.05044331]], dtype=float32)

In [14]:
# get prediction probability
prediction_prob = prediction[0][0]
prediction_prob

0.050443307

In [15]:
if prediction_prob > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
